# Notebook 05: PPO for Language Models

**Sprint 1 of the RLHF Toolkit** | Frontier AI Lab Interview Preparation

---

This notebook bridges reinforcement learning and language model fine-tuning. We implement PPO from scratch for LM alignment, then compare with the TRL library.

**Prerequisites**: Notebooks 01-04 (SFT basics, reward modeling concepts)

**Runtime**: GPU required (Colab T4 or RunPod). ~30 min for full run.

---
## 1. Self-Quiz (Active Recall)

Before reading any code, try to answer these from memory:

1. **In the RL formulation of LM fine-tuning, what is the policy? The action space? The state?**
2. **What reward signal does PPO optimize in RLHF?**
3. **Why do we add a KL penalty to the reward? What would happen without it?**
4. **What is the clipped surrogate objective in PPO, and why does clipping help?**
5. **What is a value head, and why does PPO need one?**
6. **What is GAE (Generalized Advantage Estimation), and what tradeoff does lambda control?**

<details>
<summary>Click to reveal answers after attempting</summary>

1. Policy = LM with parameters theta (maps state to distribution over next token). Action = next token from vocab. State = prompt + tokens generated so far.
2. R(x, y) = R_rm(x, y) - beta * KL(pi_theta || pi_ref), where R_rm is the reward model score.
3. KL penalty prevents the policy from diverging too far from the SFT model. Without it: reward hacking (model finds degenerate high-reward outputs) and mode collapse (model loses diversity).
4. L_clip = min(r_t * A_t, clip(r_t, 1-eps, 1+eps) * A_t). Clipping prevents destructively large policy updates.
5. Value head: a linear layer on top of LM hidden states that predicts expected future reward. Needed for advantage estimation (A = R - V).
6. GAE: A_t = sum_{l=0}^{T-t} (gamma*lambda)^l * delta_{t+l}, where delta_t = r_t + gamma*V(s_{t+1}) - V(s_t). Lambda controls bias-variance: lambda=0 is low variance/high bias (just TD error), lambda=1 is high variance/low bias (Monte Carlo-like).

</details>

---
## 2. Setup

In [ ]:
!pip install -q torch transformers datasets trl accelerate matplotlib numpy

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
import matplotlib.pyplot as plt
import numpy as np
from copy import deepcopy
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

---
## 3. RL-to-LM Mapping

The core insight of RLHF is recasting language generation as a reinforcement learning problem:

| RL Concept | LM Equivalent | Details |
|---|---|---|
| **Policy** $\pi_\theta$ | Language model with parameters $\theta$ | Maps state to distribution over next token |
| **Action** $a_t$ | Next token $y_t$ | From vocabulary $\mathcal{V}$ (typically 32K-100K actions) |
| **State** $s_t$ | Prompt $x$ + tokens generated so far $y_{<t}$ | Grows with each step |
| **Reward** $r$ | Reward model score $R(x, y)$ | Given only at end of episode (sparse) |
| **Episode** | One complete generation | From first token to EOS |
| **Environment** | The scoring function (RM) | Deterministic given (prompt, response) |
| **Value function** $V(s)$ | Expected reward from current partial generation | Learned by value head |

Key differences from typical RL:
- **Enormous action space**: |V| ~ 32K-100K (vs. ~10 in Atari)
- **Sparse reward**: Only at end of generation (no per-token reward from RM)
- **Pre-trained policy**: We start from a strong SFT model, not random
- **KL constraint**: We want to stay close to the reference policy

In [ ]:
# Concrete example: see the RL-LM mapping in action

model_name = "gpt2"  # Small enough for any GPU
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()

# The "state" is a prompt
prompt = "The meaning of life is"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
print(f"Initial state (prompt): '{prompt}'")
print(f"State as token IDs: {input_ids[0].tolist()}")
print(f"Vocabulary size (action space): {tokenizer.vocab_size}")
print()

In [ ]:
# The policy maps state -> distribution over actions (next token)
with torch.no_grad():
    outputs = model(input_ids)
    logits = outputs.logits[:, -1, :]  # Logits for next token
    probs = F.softmax(logits, dim=-1)

# Show top-10 actions (tokens) the policy would take
top_probs, top_ids = torch.topk(probs, 10)
print("Policy pi(a|s) -- Top 10 next-token probabilities:")
print("-" * 50)
for prob, tok_id in zip(top_probs[0], top_ids[0]):
    token = tokenizer.decode(tok_id)
    print(f"  Action: '{token}' (id={tok_id.item()})  ->  pi(a|s) = {prob.item():.4f}")

print(f"\nTotal probability mass in top 10: {top_probs.sum().item():.4f}")
print(f"Entropy of policy at this state: {-(probs * torch.log(probs + 1e-10)).sum().item():.2f} nats")

In [ ]:
# An "episode" = generating until EOS, collecting (state, action, log_prob) at each step

def run_episode(model, tokenizer, prompt, max_new_tokens=20, temperature=1.0):
    """Run one RL episode: generate tokens and collect trajectory data."""
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    prompt_len = input_ids.shape[1]
    
    trajectory = []  # List of (state, action, log_prob)
    current_ids = input_ids
    
    with torch.no_grad():
        for step in range(max_new_tokens):
            outputs = model(current_ids)
            logits = outputs.logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            
            # Sample action from policy
            action = torch.multinomial(probs, 1)  # Sample one token
            log_prob = torch.log(probs[0, action[0, 0]])
            
            state_text = tokenizer.decode(current_ids[0])
            action_text = tokenizer.decode(action[0])
            
            trajectory.append({
                'step': step,
                'state': state_text,
                'action': action_text,
                'action_id': action[0, 0].item(),
                'log_prob': log_prob.item(),
            })
            
            current_ids = torch.cat([current_ids, action], dim=1)
            
            if action[0, 0].item() == tokenizer.eos_token_id:
                break
    
    full_response = tokenizer.decode(current_ids[0][prompt_len:])
    return trajectory, full_response

trajectory, response = run_episode(model, tokenizer, prompt, max_new_tokens=15)

print(f"Prompt: '{prompt}'")
print(f"Generated response: '{response}'")
print(f"Episode length: {len(trajectory)} steps")
print(f"\nTrajectory (state -> action -> log_prob):")
print("-" * 70)
for t in trajectory:
    print(f"  Step {t['step']:2d}: ... '{t['state'][-30:]}' -> '{t['action']}' (log_prob={t['log_prob']:.3f})")

---
## 4. Value Head

PPO needs a **value function** $V(s_t)$ that estimates the expected cumulative reward from state $s_t$. For LMs, we add a **value head** on top of the transformer's hidden states.

Architecture:
```
Input tokens -> Transformer -> Hidden states -> {
    LM Head -> next-token logits (policy)
    Value Head -> scalar value estimate V(s)
}
```

This dual-head architecture means we share the transformer backbone between the policy and value function -- efficient but can create optimization conflicts.

In [ ]:
class ValueHead(nn.Module):
    """Value head for estimating V(s) from transformer hidden states.
    
    Typical design choices:
    - Single linear layer (simplest, used in TRL)
    - 2-layer MLP with tanh (used in some papers for stability)
    - Dropout for regularization (value head can overfit quickly)
    """
    def __init__(self, hidden_size: int, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.summary = nn.Linear(hidden_size, 1)
        # Initialize close to zero so initial values are near 0
        nn.init.zeros_(self.summary.bias)
        nn.init.normal_(self.summary.weight, std=1e-2)
    
    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        """Map hidden states to scalar value estimates.
        
        Args:
            hidden_states: (batch, seq_len, hidden_size) from transformer
        Returns:
            values: (batch, seq_len) scalar value at each position
        """
        output = self.dropout(hidden_states)
        output = self.summary(output).squeeze(-1)  # (batch, seq_len)
        return output


class CausalLMWithValueHead(nn.Module):
    """Language model with a value head for PPO.
    
    This is the dual-head architecture:
    - The base LM provides the policy (next-token probabilities)
    - The value head provides V(s) estimates for advantage computation
    - They share the transformer backbone
    """
    def __init__(self, model_name: str):
        super().__init__()
        self.pretrained_model = AutoModelForCausalLM.from_pretrained(model_name)
        self.config = self.pretrained_model.config
        self.v_head = ValueHead(self.config.n_embd)  # GPT-2 uses n_embd
    
    def forward(self, input_ids, attention_mask=None):
        """Forward pass returning both logits and values."""
        outputs = self.pretrained_model(
            input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        # Policy output: logits for next token prediction
        logits = outputs.logits  # (batch, seq_len, vocab_size)
        
        # Value output: scalar value at each position
        hidden_states = outputs.hidden_states[-1]  # Last layer hidden states
        values = self.v_head(hidden_states)  # (batch, seq_len)
        
        return logits, values
    
    def generate(self, input_ids, **kwargs):
        """Generate using the base LM (value head not used during generation)."""
        return self.pretrained_model.generate(input_ids, **kwargs)


# Instantiate and inspect
policy_model = CausalLMWithValueHead("gpt2").to(device)

# Count parameters
total_params = sum(p.numel() for p in policy_model.parameters())
vhead_params = sum(p.numel() for p in policy_model.v_head.parameters())
print(f"Total parameters: {total_params:,}")
print(f"Value head parameters: {vhead_params:,} ({100*vhead_params/total_params:.2f}%)")
print(f"Base LM parameters: {total_params - vhead_params:,}")
print(f"\nValue head is tiny compared to the base model!")

In [ ]:
# Demo: forward pass through dual-head model
test_input = tokenizer("The meaning of life is", return_tensors="pt").to(device)

with torch.no_grad():
    logits, values = policy_model(**test_input)

print(f"Input shape: {test_input['input_ids'].shape}")
print(f"Logits shape: {logits.shape}  (batch, seq_len, vocab_size)")
print(f"Values shape: {values.shape}  (batch, seq_len)")
print(f"\nValue estimates at each position:")
tokens = tokenizer.convert_ids_to_tokens(test_input['input_ids'][0])
for tok, val in zip(tokens, values[0]):
    print(f"  After '{tok}': V(s) = {val.item():.4f}")
print(f"\nNote: Values are near 0 because value head is freshly initialized.")
print(f"After PPO training, these should reflect expected reward from each state.")

---
## 5. KL Penalty

The KL penalty is crucial in RLHF. Without it, PPO would aggressively optimize the reward model, leading to:

1. **Reward hacking**: The model finds degenerate outputs that get high RM scores but are actually bad (the RM is imperfect)
2. **Mode collapse**: The model converges to a single high-reward response pattern, losing diversity
3. **Language degradation**: The model may produce grammatically broken or repetitive text

The modified reward is:

$$R_{\text{total}}(x, y) = R_{\text{RM}}(x, y) - \beta \cdot \text{KL}(\pi_\theta \| \pi_{\text{ref}})$$

where $\text{KL}(\pi_\theta \| \pi_{\text{ref}}) = \sum_t \log \frac{\pi_\theta(y_t | s_t)}{\pi_{\text{ref}}(y_t | s_t)}$

This per-token KL is computed on the actual generated tokens, not over the full distribution.

In [ ]:
def compute_kl_penalty(
    logprobs_policy: torch.Tensor,   # (batch, seq_len) - log probs under current policy
    logprobs_ref: torch.Tensor,      # (batch, seq_len) - log probs under reference model
    kl_type: str = "kl"              # "kl" or "abs" or "mse" or "full"
) -> torch.Tensor:
    """Compute per-token KL penalty between policy and reference.
    
    Several KL estimators are used in practice:
    - 'kl': standard KL = log(pi/pi_ref) = logprob_policy - logprob_ref
    - 'abs': |log(pi/pi_ref)| -- symmetric, used by some implementations
    - 'mse': (log(pi/pi_ref))^2 -- squared penalty, smoother gradients
    - 'full': full KL over distributions (more expensive but exact)
    """
    log_ratio = logprobs_policy - logprobs_ref
    
    if kl_type == "kl":
        return log_ratio  # E[log(pi/pi_ref)] -- per-token KL contribution
    elif kl_type == "abs":
        return log_ratio.abs()
    elif kl_type == "mse":
        return 0.5 * log_ratio.pow(2)
    else:
        raise ValueError(f"Unknown KL type: {kl_type}")


def get_logprobs_for_tokens(
    model: nn.Module,
    input_ids: torch.Tensor,
    response_start: int,
    return_values: bool = False
) -> torch.Tensor:
    """Get log-probabilities of response tokens under a model.
    
    This is a key operation in PPO:
    - Used to compute pi_theta(a|s) for the policy ratio
    - Used to compute pi_ref(a|s) for the KL penalty
    """
    with torch.no_grad():
        if isinstance(model, CausalLMWithValueHead):
            logits, values = model(input_ids)
        else:
            outputs = model(input_ids)
            logits = outputs.logits
            values = None
    
    # Log-probs of the actual generated tokens
    # logits[:, t, :] predicts token at position t+1
    # So for response tokens starting at response_start, we need logits at response_start-1
    log_probs = F.log_softmax(logits, dim=-1)
    
    # Gather log probs of actual tokens
    response_ids = input_ids[:, response_start:]  # (batch, response_len)
    response_logprobs = log_probs[:, response_start-1:-1, :]  # Shift to align
    
    # Gather: get log prob of each actual token
    token_logprobs = response_logprobs.gather(-1, response_ids.unsqueeze(-1)).squeeze(-1)
    
    if return_values and values is not None:
        response_values = values[:, response_start-1:-1]  # Same alignment
        return token_logprobs, response_values
    return token_logprobs


# Demo: KL between original GPT-2 and our policy model (should be ~0 since same weights)
ref_model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
ref_model.eval()

prompt = "Explain the theory of relativity in simple terms:"
tokens = tokenizer(prompt, return_tensors="pt").to(device)

# Generate a response with the policy
with torch.no_grad():
    gen_ids = policy_model.generate(
        tokens["input_ids"],
        max_new_tokens=30,
        do_sample=True,
        temperature=0.8,
        pad_token_id=tokenizer.eos_token_id
    )

prompt_len = tokens["input_ids"].shape[1]
response = tokenizer.decode(gen_ids[0][prompt_len:], skip_special_tokens=True)
print(f"Prompt: {prompt}")
print(f"Response: {response}")
print()

# Compute log probs under both models
policy_logprobs = get_logprobs_for_tokens(policy_model, gen_ids, prompt_len)
ref_logprobs = get_logprobs_for_tokens(ref_model, gen_ids, prompt_len)

kl_per_token = compute_kl_penalty(policy_logprobs, ref_logprobs, kl_type="kl")

print(f"Per-token KL divergence (should be ~0 for identical models):")
response_tokens = tokenizer.convert_ids_to_tokens(gen_ids[0][prompt_len:])
for tok, kl in zip(response_tokens, kl_per_token[0]):
    print(f"  '{tok}': KL = {kl.item():.6f}")
print(f"\nMean KL per token: {kl_per_token.mean().item():.6f}")
print(f"(Near zero because policy and ref have the same weights)")

In [ ]:
# What happens without KL penalty? Let's simulate.
# We'll create a mock reward that rewards short responses.
# Without KL: model degenerates to producing a single short token.
# With KL: model produces shorter responses but stays coherent.

print("=" * 60)
print("Thought Experiment: Effect of KL Penalty")
print("=" * 60)

print("""
Scenario: Reward = -length (prefer short responses)

WITHOUT KL penalty (beta=0):
  - The model learns to always output EOS immediately
  - Every response is empty
  - Reward is maximized but responses are useless
  - This is REWARD HACKING: the model finds a degenerate optimum

WITH KL penalty (beta=0.1):
  - The model is pulled toward shorter responses
  - BUT also pulled toward the reference model's behavior
  - Result: responses are somewhat shorter but still coherent
  - The KL penalty acts as a regularizer

WITH too HIGH KL penalty (beta=10):
  - KL penalty dominates the reward
  - Model barely changes from the reference
  - Responses are same length as before
  - The model effectively ignores the reward signal

Interview insight: The KL coefficient beta is one of the most important
hyperparameters in RLHF. The adaptive KL controller -- introduced by
Ziegler et al. 2019 (OpenAI) -- adjusts beta to maintain a target KL divergence.
""")

---
## 6. PPO Trainer from Scratch

Now we implement the full PPO training loop for language models. The key components:

1. **Generation**: Sample responses from current policy
2. **Scoring**: Compute rewards (RM score - KL penalty)
3. **Advantage estimation**: GAE on the per-token rewards
4. **PPO update**: Clipped surrogate loss + value loss

Our toy task: **reward = -length**. The model should learn to produce shorter (but still valid) responses.

In [ ]:
@dataclass
class PPOConfig:
    """PPO hyperparameters for LM fine-tuning."""
    # PPO core
    ppo_epochs: int = 4              # Number of PPO update epochs per batch
    clip_range: float = 0.2          # PPO clipping parameter epsilon
    clip_range_vf: float = 0.2       # Value function clipping (None to disable)
    vf_coef: float = 0.1             # Value loss coefficient
    
    # KL penalty
    kl_coef: float = 0.1             # KL penalty coefficient beta
    target_kl: float = 6.0           # Target KL for adaptive controller
    
    # GAE
    gamma: float = 1.0               # Discount factor (1.0 for episodic tasks)
    lam: float = 0.95                # GAE lambda
    
    # Generation
    max_new_tokens: int = 48         # Max response length
    temperature: float = 1.0         # Sampling temperature
    top_k: int = 0                   # Top-k sampling (0 = disabled)
    top_p: float = 1.0               # Nucleus sampling
    
    # Training
    learning_rate: float = 1.41e-5   # TRL's historical default, inherited from Ziegler et al. 2019; InstructGPT used ~9e-6 for its 175B PPO run
    batch_size: int = 8              # Number of prompts per batch
    mini_batch_size: int = 4         # Mini-batch size for PPO updates
    max_grad_norm: float = 0.5       # Gradient clipping
    
    # Entropy bonus (encourages exploration)
    entropy_coef: float = 0.01       # Entropy bonus coefficient

In [ ]:
class PPOTrainerFromScratch:
    """PPO trainer for language models, implemented from scratch.
    
    This is a teaching implementation that makes each step explicit.
    Production implementations (like TRL) add many optimizations.
    """
    
    def __init__(
        self,
        config: PPOConfig,
        policy_model: CausalLMWithValueHead,
        ref_model: nn.Module,
        tokenizer,
        reward_fn,  # callable(input_ids, response_ids) -> float
    ):
        self.config = config
        self.policy = policy_model
        self.ref_model = ref_model
        self.tokenizer = tokenizer
        self.reward_fn = reward_fn
        
        # Freeze reference model
        for param in self.ref_model.parameters():
            param.requires_grad = False
        
        # Optimizer -- only optimize policy (which includes value head)
        self.optimizer = Adam(self.policy.parameters(), lr=config.learning_rate)
        
        # Logging
        self.stats_history = []
    
    @torch.no_grad()
    def generate_batch(self, prompts: List[str]) -> Dict:
        """Step 1: Generate responses from current policy."""
        self.policy.eval()
        
        all_input_ids = []
        all_response_ids = []
        all_gen_ids = []
        prompt_lengths = []
        
        for prompt in prompts:
            input_ids = self.tokenizer.encode(prompt, return_tensors="pt").to(device)
            prompt_len = input_ids.shape[1]
            
            gen_ids = self.policy.generate(
                input_ids,
                max_new_tokens=self.config.max_new_tokens,
                do_sample=True,
                temperature=self.config.temperature,
                top_k=self.config.top_k if self.config.top_k > 0 else None,
                top_p=self.config.top_p,
                pad_token_id=self.tokenizer.eos_token_id,
            )
            
            all_gen_ids.append(gen_ids[0])
            all_input_ids.append(input_ids[0])
            all_response_ids.append(gen_ids[0][prompt_len:])
            prompt_lengths.append(prompt_len)
        
        return {
            'prompts': prompts,
            'gen_ids': all_gen_ids,
            'input_ids': all_input_ids,
            'response_ids': all_response_ids,
            'prompt_lengths': prompt_lengths,
        }
    
    @torch.no_grad()
    def compute_rewards_and_values(self, batch: Dict) -> Dict:
        """Step 2: Score responses and compute per-token rewards."""
        self.policy.eval()
        
        all_rewards = []
        all_kl_penalties = []
        all_rm_rewards = []
        all_values = []
        all_old_logprobs = []
        
        for i, gen_ids in enumerate(batch['gen_ids']):
            gen_ids_2d = gen_ids.unsqueeze(0)
            prompt_len = batch['prompt_lengths'][i]
            response_len = len(batch['response_ids'][i])
            
            if response_len == 0:
                # Handle empty response edge case
                all_rewards.append(torch.tensor([0.0], device=device))
                all_kl_penalties.append(torch.tensor([0.0], device=device))
                all_rm_rewards.append(0.0)
                all_values.append(torch.tensor([0.0], device=device))
                all_old_logprobs.append(torch.tensor([0.0], device=device))
                continue
            
            # Get log probs and values from policy
            logits, values = self.policy(gen_ids_2d)
            log_probs = F.log_softmax(logits, dim=-1)
            
            # Response token log probs under policy
            resp_ids = gen_ids[prompt_len:]  # (response_len,)
            resp_logprobs = log_probs[0, prompt_len-1:-1, :]  # Align: logits at t predict t+1
            old_logprobs = resp_logprobs.gather(-1, resp_ids.unsqueeze(-1)).squeeze(-1)
            
            # Response values
            resp_values = values[0, prompt_len-1:-1]
            
            # Reference model log probs
            ref_outputs = self.ref_model(gen_ids_2d)
            ref_log_probs = F.log_softmax(ref_outputs.logits, dim=-1)
            ref_resp_logprobs = ref_log_probs[0, prompt_len-1:-1, :]
            ref_token_logprobs = ref_resp_logprobs.gather(-1, resp_ids.unsqueeze(-1)).squeeze(-1)
            
            # KL penalty per token
            kl_per_token = old_logprobs - ref_token_logprobs
            
            # External reward from reward function (scalar for full response)
            rm_reward = self.reward_fn(
                batch['input_ids'][i],
                batch['response_ids'][i]
            )
            
            # Per-token rewards: KL penalty at each token, RM reward at last token
            per_token_rewards = -self.config.kl_coef * kl_per_token
            per_token_rewards[-1] += rm_reward  # Add RM reward to last token
            
            all_rewards.append(per_token_rewards)
            all_kl_penalties.append(kl_per_token)
            all_rm_rewards.append(rm_reward)
            all_values.append(resp_values)
            all_old_logprobs.append(old_logprobs)
        
        batch['rewards'] = all_rewards
        batch['kl_penalties'] = all_kl_penalties
        batch['rm_rewards'] = all_rm_rewards
        batch['values'] = all_values
        batch['old_logprobs'] = all_old_logprobs
        return batch
    
    @torch.no_grad()
    def compute_advantages(self, batch: Dict) -> Dict:
        """Step 3: Compute advantages using GAE (Generalized Advantage Estimation).
        
        GAE: A_t = sum_{l=0}^{T-t} (gamma * lambda)^l * delta_{t+l}
        where delta_t = r_t + gamma * V(s_{t+1}) - V(s_t)
        """
        all_advantages = []
        all_returns = []
        
        for rewards, values in zip(batch['rewards'], batch['values']):
            T = len(rewards)
            advantages = torch.zeros(T, device=device)
            last_gae = 0
            
            for t in reversed(range(T)):
                if t == T - 1:
                    next_value = 0  # Terminal state
                else:
                    next_value = values[t + 1]
                
                delta = rewards[t] + self.config.gamma * next_value - values[t]
                advantages[t] = last_gae = delta + self.config.gamma * self.config.lam * last_gae
            
            returns = advantages + values  # Returns = advantages + values
            all_advantages.append(advantages)
            all_returns.append(returns)
        
        batch['advantages'] = all_advantages
        batch['returns'] = all_returns
        return batch
    
    def ppo_update(self, batch: Dict) -> Dict:
        """Step 4: PPO clipped surrogate loss + value loss.
        
        This runs multiple epochs of updates on the same batch.
        """
        self.policy.train()
        
        total_pg_loss = 0
        total_vf_loss = 0
        total_entropy = 0
        total_clipfrac = 0
        n_updates = 0
        
        for epoch in range(self.config.ppo_epochs):
            for i in range(len(batch['gen_ids'])):
                gen_ids = batch['gen_ids'][i].unsqueeze(0)
                prompt_len = batch['prompt_lengths'][i]
                resp_ids = batch['response_ids'][i]
                response_len = len(resp_ids)
                
                if response_len == 0:
                    continue
                
                old_logprobs = batch['old_logprobs'][i]
                advantages = batch['advantages'][i]
                returns = batch['returns'][i]
                old_values = batch['values'][i]
                
                # Normalize advantages (important for stability!)
                if len(advantages) > 1:
                    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
                
                # Forward pass through current policy
                logits, values = self.policy(gen_ids)
                log_probs = F.log_softmax(logits, dim=-1)
                
                # Current log probs of response tokens
                curr_resp_logprobs = log_probs[0, prompt_len-1:-1, :]
                curr_token_logprobs = curr_resp_logprobs.gather(
                    -1, resp_ids.unsqueeze(-1)
                ).squeeze(-1)
                
                # Current values
                curr_values = values[0, prompt_len-1:-1]
                
                # Policy ratio
                ratio = torch.exp(curr_token_logprobs - old_logprobs)
                
                # Clipped surrogate loss
                pg_loss1 = -advantages * ratio
                pg_loss2 = -advantages * torch.clamp(
                    ratio, 1.0 - self.config.clip_range, 1.0 + self.config.clip_range
                )
                pg_loss = torch.max(pg_loss1, pg_loss2).mean()
                
                # Value loss (with optional clipping)
                if self.config.clip_range_vf is not None:
                    clipped_values = old_values + torch.clamp(
                        curr_values - old_values,
                        -self.config.clip_range_vf,
                        self.config.clip_range_vf
                    )
                    vf_loss1 = (curr_values - returns) ** 2
                    vf_loss2 = (clipped_values - returns) ** 2
                    vf_loss = 0.5 * torch.max(vf_loss1, vf_loss2).mean()
                else:
                    vf_loss = 0.5 * ((curr_values - returns) ** 2).mean()
                
                # Entropy bonus (encourages exploration)
                probs = F.softmax(logits[0, prompt_len-1:-1, :], dim=-1)
                entropy = -(probs * (probs + 1e-10).log()).sum(-1).mean()
                
                # Total loss
                loss = pg_loss + self.config.vf_coef * vf_loss - self.config.entropy_coef * entropy
                
                # Update
                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.policy.parameters(), self.config.max_grad_norm)
                self.optimizer.step()
                
                # Stats
                clip_fraction = (torch.abs(ratio - 1.0) > self.config.clip_range).float().mean()
                total_pg_loss += pg_loss.item()
                total_vf_loss += vf_loss.item()
                total_entropy += entropy.item()
                total_clipfrac += clip_fraction.item()
                n_updates += 1
        
        n_updates = max(n_updates, 1)
        return {
            'pg_loss': total_pg_loss / n_updates,
            'vf_loss': total_vf_loss / n_updates,
            'entropy': total_entropy / n_updates,
            'clip_fraction': total_clipfrac / n_updates,
        }
    
    def train_step(self, prompts: List[str]) -> Dict:
        """One full PPO training step: generate -> score -> advantage -> update."""
        # Step 1: Generate
        batch = self.generate_batch(prompts)
        
        # Step 2: Score and compute rewards
        batch = self.compute_rewards_and_values(batch)
        
        # Step 3: Compute advantages
        batch = self.compute_advantages(batch)
        
        # Step 4: PPO update
        update_stats = self.ppo_update(batch)
        
        # Aggregate stats
        mean_rm_reward = np.mean([r if isinstance(r, float) else r.item()
                                  for r in batch['rm_rewards']])
        mean_kl = np.mean([kl.mean().item() for kl in batch['kl_penalties']])
        mean_response_len = np.mean([len(r) for r in batch['response_ids']])
        
        stats = {
            'mean_reward': mean_rm_reward,
            'mean_kl': mean_kl,
            'mean_response_len': mean_response_len,
            **update_stats,
        }
        self.stats_history.append(stats)
        return stats

In [ ]:
# Toy task: reward = -length (model should learn to be more concise)
# This is a good test because:
# 1. Easy to verify: we can measure response length
# 2. Clear signal: shorter = better
# 3. Shows KL tradeoff: without KL, model would just output EOS

def length_penalty_reward(input_ids, response_ids):
    """Negative length reward: shorter responses score higher."""
    length = len(response_ids)
    # Normalize to roughly [-1, 1] range
    return -length / 20.0  # Max reward 0 (empty), min ~ -2.4 for 48 tokens

# Training prompts (diverse to prevent overfitting)
train_prompts = [
    "Explain quantum computing:",
    "What is machine learning?",
    "Describe the solar system:",
    "How does the internet work?",
    "What causes earthquakes?",
    "Explain photosynthesis:",
    "What is artificial intelligence?",
    "How do airplanes fly?",
    "What is climate change?",
    "Explain how vaccines work:",
    "What is the theory of evolution?",
    "How does electricity work?",
    "What is DNA?",
    "Explain the water cycle:",
    "What causes thunder and lightning?",
    "How do computers process information?",
]

In [ ]:
# Initialize trainer
config = PPOConfig(
    ppo_epochs=2,
    batch_size=4,
    max_new_tokens=48,
    kl_coef=0.1,
    learning_rate=1.41e-5,
    temperature=0.8,
    clip_range=0.2,
    entropy_coef=0.01,
)

# Fresh policy and reference
policy_model = CausalLMWithValueHead("gpt2").to(device)
ref_model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
ref_model.eval()

trainer = PPOTrainerFromScratch(
    config=config,
    policy_model=policy_model,
    ref_model=ref_model,
    tokenizer=tokenizer,
    reward_fn=length_penalty_reward,
)

print("Trainer initialized. Starting training loop...")
print(f"Config: {config}")

In [ ]:
# Training loop
import random

NUM_STEPS = 40  # Number of PPO steps

print("Training PPO on toy task (reward = -length)...")
print("=" * 70)

for step in range(NUM_STEPS):
    # Sample a batch of prompts
    batch_prompts = random.sample(train_prompts, min(config.batch_size, len(train_prompts)))
    
    # One PPO step
    stats = trainer.train_step(batch_prompts)
    
    if step % 5 == 0 or step == NUM_STEPS - 1:
        print(f"Step {step:3d} | "
              f"Reward: {stats['mean_reward']:+.3f} | "
              f"KL: {stats['mean_kl']:.4f} | "
              f"Resp Len: {stats['mean_response_len']:.1f} | "
              f"PG Loss: {stats['pg_loss']:.4f} | "
              f"VF Loss: {stats['vf_loss']:.4f} | "
              f"Entropy: {stats['entropy']:.2f} | "
              f"Clip: {stats['clip_fraction']:.3f}")

print("\nTraining complete!")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('PPO Training Curves (Toy Task: reward = -length)', fontsize=14)

history = trainer.stats_history
steps = range(len(history))

# Reward
axes[0, 0].plot(steps, [s['mean_reward'] for s in history], 'b-')
axes[0, 0].set_title('Mean Reward')
axes[0, 0].set_xlabel('Step')
axes[0, 0].set_ylabel('Reward')
axes[0, 0].grid(True, alpha=0.3)

# KL divergence
axes[0, 1].plot(steps, [s['mean_kl'] for s in history], 'r-')
axes[0, 1].set_title('Mean KL Divergence')
axes[0, 1].set_xlabel('Step')
axes[0, 1].set_ylabel('KL')
axes[0, 1].grid(True, alpha=0.3)

# Response length
axes[0, 2].plot(steps, [s['mean_response_len'] for s in history], 'g-')
axes[0, 2].set_title('Mean Response Length')
axes[0, 2].set_xlabel('Step')
axes[0, 2].set_ylabel('Tokens')
axes[0, 2].grid(True, alpha=0.3)

# Policy loss
axes[1, 0].plot(steps, [s['pg_loss'] for s in history], 'purple')
axes[1, 0].set_title('Policy Gradient Loss')
axes[1, 0].set_xlabel('Step')
axes[1, 0].grid(True, alpha=0.3)

# Value loss
axes[1, 1].plot(steps, [s['vf_loss'] for s in history], 'orange')
axes[1, 1].set_title('Value Function Loss')
axes[1, 1].set_xlabel('Step')
axes[1, 1].grid(True, alpha=0.3)

# Entropy
axes[1, 2].plot(steps, [s['entropy'] for s in history], 'teal')
axes[1, 2].set_title('Policy Entropy')
axes[1, 2].set_xlabel('Step')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey observations:")
print("- Reward should increase (less negative = shorter responses)")
print("- KL should increase but stay bounded (policy is changing but KL penalty constrains it)")
print("- Response length should decrease (the model is learning the reward signal)")
print("- Entropy may decrease (model becomes more certain about shorter responses)")

In [ ]:
# Compare responses: before vs after PPO
print("=" * 70)
print("BEFORE vs AFTER PPO Training")
print("=" * 70)

eval_prompts = [
    "Explain quantum computing:",
    "What is machine learning?",
    "Describe the solar system:",
]

# Reference model (before PPO)
for prompt in eval_prompts:
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    prompt_len = input_ids.shape[1]
    
    with torch.no_grad():
        # Before (reference model)
        ref_gen = ref_model.generate(
            input_ids, max_new_tokens=48, do_sample=True,
            temperature=0.8, pad_token_id=tokenizer.eos_token_id
        )
        ref_response = tokenizer.decode(ref_gen[0][prompt_len:], skip_special_tokens=True)
        
        # After (PPO-trained policy)
        ppo_gen = policy_model.generate(
            input_ids, max_new_tokens=48, do_sample=True,
            temperature=0.8, pad_token_id=tokenizer.eos_token_id
        )
        ppo_response = tokenizer.decode(ppo_gen[0][prompt_len:], skip_special_tokens=True)
    
    print(f"\nPrompt: {prompt}")
    print(f"  BEFORE ({len(ref_gen[0])-prompt_len} tokens): {ref_response[:120]}")
    print(f"  AFTER  ({len(ppo_gen[0])-prompt_len} tokens): {ppo_response[:120]}")
    print("-" * 70)

**Insider Tip:** The generation bottleneck is the #1 engineering challenge in RLHF. At scale, 80-90% of training time is spent on generation, not gradient updates. This is because generation is autoregressive (sequential token-by-token), while training can be parallelized across tokens. Solutions include: speculative decoding during generation, async generation pipelines (generate next batch while updating on current batch), vLLM/TGI for optimized inference, and frameworks like OpenRLHF and veRL that separate generation from training across different GPU groups. If asked "what's the hardest part of scaling RLHF?" in an interview, this is the answer that will impress.

---
## 7. Using TRL's PPOTrainer

TRL (Transformer Reinforcement Learning) is Hugging Face's library for RLHF. Let's compare our from-scratch implementation with TRL's `PPOTrainer`.

Key differences from our implementation:
- TRL handles padding/batching automatically
- TRL has optimized implementations of GAE, KL computation
- TRL supports distributed training, gradient accumulation
- TRL has adaptive KL controllers

**API Note (2024-2025):** TRL's API has changed significantly across versions. The legacy `PPOTrainer` API was available through ~v0.11 and was reworked in v0.12 (late 2024). The code below uses the legacy (pre-v0.12) API style. If you are using a newer version, consult the TRL docs at https://huggingface.co/docs/trl. For the latest TRL versions, consider using `OnlineDPOTrainer` or the updated PPO API. The from-scratch implementation in Section 6 above remains the authoritative reference regardless of library version changes.

In [ ]:
from trl import PPOTrainer, PPOConfig as TRLPPOConfig, AutoModelForCausalLMWithValueHead
from trl.core import LengthSampler

# TRL PPO Config
trl_config = TRLPPOConfig(
    model_name="gpt2",
    learning_rate=1.41e-5,
    batch_size=4,
    mini_batch_size=2,
    ppo_epochs=2,
    log_with=None,  # No wandb for this demo
)

# TRL provides its own value head model
trl_model = AutoModelForCausalLMWithValueHead.from_pretrained("gpt2").to(device)
trl_ref_model = AutoModelForCausalLMWithValueHead.from_pretrained("gpt2").to(device)
trl_tokenizer = AutoTokenizer.from_pretrained("gpt2")
trl_tokenizer.pad_token = trl_tokenizer.eos_token

# Create TRL trainer
trl_trainer = PPOTrainer(
    config=trl_config,
    model=trl_model,
    ref_model=trl_ref_model,
    tokenizer=trl_tokenizer,
)

print("TRL PPO Trainer initialized.")
print(f"\nKey config options that matter:")
print(f"  init_kl_coef: {trl_config.init_kl_coef} (initial KL coefficient)")
print(f"  target: {trl_config.target} (target KL for adaptive controller)")
print(f"  clip_range: Not directly in TRLPPOConfig; uses cliprange param if available")
print(f"  ppo_epochs: {trl_config.ppo_epochs} (PPO update epochs per batch)")

In [ ]:
# Run a few TRL training steps with same reward function
trl_stats = []

print("Training with TRL PPOTrainer (same reward = -length)...")
print("=" * 70)

for step in range(10):
    # Prepare batch of queries
    batch_prompts = random.sample(train_prompts, trl_config.batch_size)
    query_tensors = [trl_tokenizer.encode(p, return_tensors="pt").squeeze().to(device)
                     for p in batch_prompts]
    
    # Generate responses
    response_tensors = []
    for query in query_tensors:
        gen = trl_trainer.generate(query.unsqueeze(0), max_new_tokens=48,
                                   do_sample=True, temperature=0.8,
                                   pad_token_id=trl_tokenizer.eos_token_id)
        response = gen.squeeze()[len(query):]
        response_tensors.append(response)
    
    # Compute rewards
    rewards = [torch.tensor(-len(r) / 20.0, device=device) for r in response_tensors]
    
    # PPO step
    stats = trl_trainer.step(query_tensors, response_tensors, rewards)
    
    mean_reward = np.mean([r.item() for r in rewards])
    mean_len = np.mean([len(r) for r in response_tensors])
    trl_stats.append({'reward': mean_reward, 'length': mean_len})
    
    if step % 2 == 0:
        print(f"Step {step:3d} | Reward: {mean_reward:+.3f} | Resp Len: {mean_len:.1f}")

print("\nTRL training complete!")
print("\nKey advantage of TRL over from-scratch:")
print("  - Handles batching/padding automatically")
print("  - Adaptive KL controller (adjusts beta to hit target KL)")
print("  - Distributed training support")
print("  - Well-tested and maintained")

---
## 8. "Why Does This Work?" -- Critical Thinking Prompts

### Q: What if the KL coefficient is too high?

If $\beta$ is too large, the KL penalty dominates the reward signal. The model barely changes from the reference policy. You've effectively turned RLHF into a no-op. In practice, you'll see:
- KL stays near zero
- Reward barely improves
- Responses are identical to the SFT model

**Adaptive KL controllers** solve this: they adjust $\beta$ to maintain a target KL divergence (e.g., target KL = 6 nats). If KL is too low, decrease $\beta$; if too high, increase $\beta$.

### Q: What if the KL coefficient is too low?

The model aggressively optimizes the reward model, leading to **reward hacking**:
- The RM is an imperfect proxy for human preferences
- The policy finds outputs that exploit RM weaknesses
- Example: RM trained on helpful data might give high scores to long, verbose responses that repeat the question -- these aren't actually helpful
- Gao et al. (2023) "Scaling Laws for Reward Model Overoptimization" quantifies this: reward model score increases, but actual human preference *decreases* past a certain point

### Q: Why not just use supervised learning on high-reward outputs?

This is actually a valid approach called **rejection sampling fine-tuning** (or Best-of-N):
1. Generate N responses per prompt
2. Score with reward model
3. Fine-tune on the best response (supervised)

**Pros** vs PPO: Simpler, no value function, no GAE, no clipping, no instability

**Cons**: Less sample-efficient (need many samples to find good ones), doesn't directly optimize the reward (only imitates good samples), can't improve beyond what random sampling finds.

Llama-2 used rejection sampling in addition to PPO. Some teams find it sufficient.

### Q: Why PPO instead of simpler RL algorithms?

- **REINFORCE** (vanilla policy gradient): High variance, unstable for LMs
- **PPO**: Clipping prevents catastrophically large updates. Crucial when the "environment" (reward model) is noisy.
- **A2C**: Works but PPO's clipping adds robustness
- **TRPO**: Trust region is expensive to compute for LMs (requires conjugate gradient)

PPO hits the sweet spot of stability and implementation simplicity for LM fine-tuning.

---
## Interview Question Bank: PPO for Language Models

*This is where RL meets LLMs -- the core of RLHF. These questions are specifically about applying PPO to language model training, which has unique challenges compared to standard RL.*

---

### Question 1: "What are the key differences between PPO for games and PPO for language models?"

**What we're testing:** Understanding of the unique engineering and algorithmic challenges of applying RL to language generation.

**Good answer:** Identifies the major differences: (1) In LM training, the policy IS the language model -- actions are token selections from a 32K-128K vocabulary, (2) A KL penalty against a reference model is added to prevent the policy from deviating too far from the SFT model, (3) A value head is added to the language model architecture to estimate state values, (4) Generation is autoregressive, so each "step" requires a full forward pass.

**Great answer (Principal-level):** All of the above, plus: (1) **The generation bottleneck:** In games, environment simulation is fast. In RLHF, generating responses is the bottleneck -- each forward pass through a 70B model is expensive, and you need to generate full responses (50-2000 tokens) before scoring. This makes RLHF roughly 10x more expensive than SFT per training step. (2) **The 4-model memory problem:** During PPO training, you need 4 models in memory simultaneously: the active policy, the reference policy (frozen copy), the reward model, and the value function (often a head on the policy). For 70B models, this requires >1TB of GPU memory. (3) **Credit assignment:** In games, rewards are per-step. In RLHF, the reward is for the entire response (sparse reward at the end). This makes credit assignment harder -- which token made the response good or bad? (4) **Non-stationarity:** The reward model is a learned approximation, not a true environment. It can be exploited.

**Red flag:** Treats PPO for LMs as identical to PPO for games. Doesn't mention the KL penalty. Doesn't know about the generation bottleneck.

**Follow-up:** "How do you parallelize RLHF training across 1000 GPUs?" (This is a system design question. Key ideas: separate the generation phase (tensor parallel across GPUs) from the training phase (data parallel + tensor parallel). Some teams use asynchronous generation with a "rollout pool" that generates in parallel while training happens on the previous batch. Discuss how DeepSpeed-Chat and TRL handle this.)

---

### Question 2: "What is the KL penalty and why is it crucial?"

**What we're testing:** Understanding of a design choice that is specific to RLHF and critical for its success.

**Good answer:** The KL penalty $\beta \cdot D_{KL}(\pi_\theta || \pi_{ref})$ penalizes the policy for deviating from the reference (SFT) model. Without it, the policy would quickly find degenerate behaviors that exploit the reward model -- repeating high-reward phrases, generating incoherent but high-scoring text, etc. The modified reward becomes $r'(x,y) = r(x,y) - \beta \cdot \log\frac{\pi_\theta(y|x)}{\pi_{ref}(y|x)}$.

**Great answer (Principal-level):** Explains the theory and practice: (1) **Why it works:** The KL penalty creates a regularized objective. The optimal solution is not the reward-maximizing policy but a balance between reward and proximity to the reference. This is equivalent to solving $\max_\pi \mathbb{E}[r(x,y)] - \beta D_{KL}(\pi || \pi_{ref})$, whose closed-form solution is $\pi^*(y|x) \propto \pi_{ref}(y|x) \exp(r(x,y)/\beta)$ -- this is exactly the DPO connection. (2) **Adaptive KL coefficient:** In practice, $\beta$ is often adapted during training. If KL exceeds a target (e.g., 6 nats), $\beta$ is increased; if KL is below target, $\beta$ is decreased. This prevents both mode collapse (KL too high) and stagnation (KL too low). (3) **What happens without KL:** The model rapidly degenerates. It finds "adversarial examples" for the reward model -- text that scores high but is nonsensical or repetitive. (4) **Connection to DPO:** The KL-regularized RLHF objective is exactly what DPO optimizes directly, without needing to run PPO at all. Understanding this connection is essential.

**Red flag:** Can't explain why the KL penalty is needed (says something vague like "regularization"). Doesn't know the connection between KL coefficient and reward hacking. Hasn't thought about what happens when $\beta$ is wrong.

**Follow-up 1:** "You're training with PPO and the KL divergence is increasing rapidly. What do you do?" (Increase $\beta$, reduce learning rate, check that the reward model isn't being exploited. If KL is increasing AND reward is increasing but human eval says quality is decreasing, you have reward hacking.)

**Follow-up 2:** "Derive the closed-form solution to the KL-regularized objective." (Tests mathematical depth. Start with the Lagrangian, use calculus of variations or the KKT conditions to get $\pi^*(y|x) = \frac{1}{Z(x)} \pi_{ref}(y|x) \exp(r(x,y)/\beta)$.)

---
## Production Implementation Notes: PPO for LMs at Frontier Scale

*The jump from PPO on CartPole to PPO on a 70B language model is arguably the biggest engineering leap in all of RLHF.*

### The Textbook vs. Reality Gap

| Component | Textbook Version (this notebook) | Production Version (frontier labs) |
|-----------|--------------------------------|-----------------------------------|
| **Policy model** | GPT-2 (124M params) | 70B-400B parameter models |
| **Generation** | Single GPU, sequential | Tensor parallel across 8 GPUs, batch generation |
| **Reward model** | Simple length-based | Separate 70B model requiring its own GPU allocation |
| **Value head** | MLP on top of GPT-2 | Separate value model or shared backbone with careful gradient isolation |
| **Training** | Single GPU, toy task | 256-1024 GPUs, FSDP + tensor parallel, weeks of training |
| **KL reference** | In-memory copy | Separate model shard, sometimes approximated with cached logprobs |

### Scale Numbers You Should Know

- **RLHF training cost:** 10-20x more expensive than SFT per training step. For a 70B model: 256-1024 H100 GPUs for 1-2 weeks -- roughly $0.5M-$2M in GPU cost at ~$2-3/H100-hr. The full program cost is substantially higher once experiments and ablations are included.
- **The 4-model memory problem:** At 70B scale in BF16, each model is ~140GB. Four models = ~560GB minimum, plus activations, optimizer states, and KV-cache for generation. This requires >80 GPUs just for memory, before any parallelism for speed.
- **Generation throughput:** The bottleneck. A 70B model generates ~50-100 tokens/second per GPU. Generating 10K responses of 500 tokens each takes hours on 8 GPUs. Teams use vLLM or TensorRT-LLM inference engines for generation, separate from the training framework.
- **PPO batch size:** Typically 256-1024 responses per PPO update. Each response is generated, scored by the RM, and used for 1-4 PPO epochs.
- **Total training steps:** 500-5000 PPO steps. Each step generates, scores, and trains on one batch.

### Engineering Challenges Not in Papers

1. **Generation-training pipeline:** Production RLHF systems separate generation from training. Generation uses an inference-optimized engine (vLLM, TGI); training uses a training framework (DeepSpeed, Megatron). Coordinating these is a distributed systems challenge.
2. **Model synchronization:** After each PPO update, the new policy weights must be sent to the generation engine. With 70B models across hundreds of GPUs, this weight transfer is a significant bottleneck.
3. **Mixed precision in RL:** Generating in FP16/BF16 and training in BF16/FP32 creates subtle numerical differences. Log probabilities computed during generation vs during training can differ, causing the importance sampling ratio to be noisy.
4. **Value head instability:** The value head often converges slowly or becomes unstable. Some teams pretrain the value head on RM scores before starting PPO. Others use a separate value model entirely.
5. **Reward model serving:** The RM must evaluate every generated response. For large batches, this means running inference on hundreds of long sequences through a 70B model. Teams batch RM inference carefully and sometimes use a smaller RM for PPO while keeping a large RM for evaluation.

### Monitoring PPO for LMs

- **KL divergence over time:** The primary safety metric. Should increase gradually. Sudden jumps indicate instability. Typical final KL: 5-15 nats from the reference model.
- **Reward vs KL curve:** Plot reward on Y-axis, KL on X-axis. The slope of this curve shows reward efficiency. A good RLHF run gets a lot of reward for little KL. A bad run (reward hacking) gets reward but the KL-reward slope flattens or the curve bends.
- **Sample inspection:** The most important metric. Regularly inspect generated samples and compare to the SFT baseline. Automated metrics can miss subtle quality degradation.
- **Per-category metrics:** Track reward and KL separately for different prompt categories (coding, math, creative writing, safety). The model may improve on some categories while degrading on others.
- **Entropy:** Policy entropy should decrease moderately. Rapid entropy collapse = mode collapse. Entropy staying too high = the model isn't learning.

---
## How This Gets Tested in Interviews

### Where PPO-for-LM Questions Appear

| Company | Round | Format | Depth |
|---------|-------|--------|-------|
| **Anthropic** | Onsite (core competency) | Deep discussion + system design | Very deep -- this is the heart of their training pipeline |
| **OpenAI** | Onsite (RLHF round) | Discussion + design | Deep -- expects practical experience or deep theoretical understanding |
| **DeepMind** | Onsite | Discussion | Moderate to deep -- more theoretical, less implementation-focused |
| **Character.AI / Cohere** | Onsite | Discussion + coding | Moderate -- practical understanding, focus on trade-offs |

### Time Expectations

- **"Explain PPO for LMs vs PPO for games"**: 10-15 minute discussion. Hit the 4 key differences (KL penalty, generation bottleneck, 4-model problem, sparse reward).
- **"Design the PPO training loop for a 70B model"**: 30-45 minute system design. Cover generation pipeline, model placement across GPUs, training loop, monitoring.
- **"The KL is increasing rapidly -- diagnose"**: 10-15 minute debugging discussion. Walk through a systematic diagnosis.

### Senior vs. Principal Expectations

**senior ML engineer:**
- Explain the PPO-for-LM setup: policy, reference, value head, reward model
- Understand the KL penalty and why it's needed
- Know the generation-training loop at a high level
- Implement a basic PPO-for-LM training step given the components

**principal ML engineer:**
- All of the above, plus:
- Design the distributed system: how to place 4 models across 1000 GPUs
- Derive the closed-form solution to the KL-regularized objective and connect it to DPO
- Discuss the generation bottleneck quantitatively: how many tokens/second, how this affects training throughput
- Know adaptive KL scheduling: what target KL to use, how to adjust $\beta$
- Reason about alternatives: when is DPO sufficient? When do you need online RL?
- Discuss reward hacking concretely: give 3 examples of degenerate behaviors and how to detect/prevent each
- Have an opinion on: is the 4-model problem the main barrier to RLHF scaling, or is it the reward model quality?

### Preparation Checklist

- [ ] Draw the PPO-for-LM training loop from memory (generation -> scoring -> advantage computation -> PPO update -> repeat)
- [ ] Be ready to explain the 4-model memory problem and how to solve it (model parallelism, offloading, separate generation/training)
- [ ] Know the KL-regularized objective and its closed-form solution (connects to DPO)
- [ ] Have a debugging playbook: "KL increasing too fast" -> "reward going up but quality going down" -> "model generating repetitive text"
- [ ] Be ready for: "How would you make RLHF 10x cheaper?" (Ideas: DPO/offline methods, smaller RM, rejection sampling instead of PPO, RLHF on smaller model then distill)

---
## 9. Flashcard Summary

| # | Question | Answer |
|---|----------|--------|
| 1 | What is the policy in RLHF? | The language model $\pi_\theta$ -- it maps (prompt + generated tokens) to a distribution over the next token. |
| 2 | What is the action space? | The vocabulary $\mathcal{V}$, typically 32K-100K tokens. Much larger than typical RL. |
| 3 | What is the state in RLHF? | The prompt concatenated with all tokens generated so far: $s_t = (x, y_1, \ldots, y_{t-1})$. |
| 4 | Why is the reward sparse? | The reward model scores the *complete* response. No per-token reward signal until the end of generation. |
| 5 | What does the value head do? | Predicts expected future reward $V(s_t)$ from the current state. Needed for advantage estimation. |
| 6 | Write the modified reward in RLHF. | $R(x,y) = R_{\text{RM}}(x,y) - \beta \cdot \text{KL}(\pi_\theta \| \pi_{\text{ref}})$ |
| 7 | What happens without KL penalty? | Reward hacking, mode collapse, language degradation. The model exploits RM weaknesses. |
| 8 | What is an adaptive KL controller? | It adjusts $\beta$ dynamically to maintain a target KL divergence. If KL too low, decrease $\beta$; if too high, increase $\beta$. |
| 9 | What is the PPO clipped surrogate objective? | $L = \min(r_t A_t, \text{clip}(r_t, 1-\epsilon, 1+\epsilon) A_t)$ where $r_t = \pi_\theta / \pi_{\text{old}}$. |
| 10 | What does GAE do and what does lambda control? | GAE estimates advantages with $A_t = \sum_l (\gamma\lambda)^l \delta_{t+l}$. $\lambda$ controls bias-variance: 0 = low variance/high bias, 1 = high variance/low bias. |
| 11 | What is rejection sampling fine-tuning? | Generate N responses, pick the best (by RM score), fine-tune on it with supervised learning. Simpler alternative to PPO. |
| 12 | Why PPO over REINFORCE for LMs? | PPO's clipping prevents catastrophically large updates. REINFORCE has high variance and is unstable for the large action spaces of LMs. |

---
## 10. Paper Guide

### Primary: "Secrets of RLHF in Large Language Models Part I: PPO"
**Zheng et al. (2023)** | https://arxiv.org/abs/2307.04964

**Why this paper matters:**
This is one of the few papers that details the practical engineering of PPO for LMs. Most RLHF papers gloss over implementation details. This one covers:

**Key contributions:**
1. **PPO stability analysis**: Systematic study of which PPO implementation details matter for stable LM training. (Note: the PRM vs ORM distinction is from Uesato et al. 2022 and Lightman et al. 2023, not this paper.)
2. **PPO-max**: A more stable variant of PPO for LMs with reward clipping and advantage normalization
3. **Reward model training tricks**: Calibration, data balancing, and filtering strategies
4. **Practical KL divergence**: Comparison of different KL estimators and their effects

**What to focus on (for interviews):**
- Section 3: PPO implementation details (what breaks and how to fix it)
- Section 4: Reward model overoptimization and mitigation
- Table 1: Effect of different PPO configurations

**Interview connection:**
If asked "How would you implement RLHF from scratch?", you should be able to describe:
1. The 4-model setup (policy, reference, reward model, value model)
2. The generation-scoring-advantage-update loop
3. Key hyperparameters (KL coef, clip range, PPO epochs, batch size)
4. Common failure modes and how to diagnose them

### Additional References:
- **Schulman et al. (2017)**: "Proximal Policy Optimization Algorithms" -- the original PPO paper
- **Ouyang et al. (2022)**: "Training language models to follow instructions with human feedback" (InstructGPT) -- first large-scale RLHF
- **Stiennon et al. (2020)**: "Learning to summarize from human feedback" -- RLHF for summarization, simpler setting

### Recent Papers (2024-2025) -- Know for Interviews:
- **OpenRLHF** (Hu et al. 2024): Open-source RLHF framework with Ray-based distributed generation -- demonstrates the generation/training separation architecture used in production
- **veRL (Volcano Engine RL)** (Sheng et al. 2024): HybridFlow architecture for efficient RLHF training, co-locating generation and training on same GPUs via dynamic memory management
- **DeepSeek-R1** (DeepSeek 2025): Uses GRPO (no value model) for reasoning model training at scale -- demonstrates that RL can teach models to reason, not just follow instructions
- **RLVR (RL with Verifiable Rewards)** (Lambert et al. 2024): Using programmatic/verifiable reward signals instead of learned reward models -- greatly reduces reward-model gaming, though verifiable rewards are still gamed in practice (unit-test exploitation, special-casing, format hacking)